# Connectivity Feature Extraction

## Imports

In [4]:
from pathlib import Path
import re
import json

import mne
import mne_connectivity

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Loading the EEG data files

In [ ]:
PROJECT_ROOT = Path.cwd()

DATASET_PATH = PROJECT_ROOT / "eeg_dataset"

EXPECTED_SUBJECTS = {
    "AD": set(range(1, 37)),
    "CN": set(range(37, 66)),
    "FTD": set(range(66, 89)),
}

# desired anterior -> posterior order
ANTERIOR_TO_POSTERIOR = [   # Frontopolar
                            "Fp1", "Fp2",
                            # Frontal
                            "F7", "F3", "Fz", "F4", "F8",
                            # Central + adjacent temporal (anterior temporal)
                            "T3", "C3", "Cz", "C4", "T4",
                            # Posterior temporal + parietal
                            "T5", "P3", "Pz", "P4", "T6",
                            # Occipital
                            "O1", "O2"
                        ]

def load_dataset(dataset_path, channel_order):
    recordings = []

    for label in EXPECTED_SUBJECTS:
        folder = dataset_path / label

        for file_path in sorted(folder.glob("*.set")):
            match = re.search(r"sub-(\d+)", file_path.name)
            subject = int(match.group(1))

            raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
            
            raw.reorder_channels(channel_order)   
         
            duration = raw.n_times / raw.info["sfreq"]

            recordings.append({
                "subject": subject,
                "label": label,
                "duration": duration,
                "data": raw,
            })

    return recordings



recordings = load_dataset(DATASET_PATH, channel_order=ANTERIOR_TO_POSTERIOR)

C:\Users\Online\AppData\Local\Temp\ipykernel_17688\372025423.py:32: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
C:\Users\Online\AppData\Local\Temp\ipykernel_17688\372025423.py:32: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
C:\Users\Online\AppData\Local\Temp\ipykernel_17688\372025423.py:32: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
C:\Users\Online\AppData\Local\Temp\ipykernel_17688\372025423.py:32: RuntimeWarning: The data contains 'boundary' events, indicating data discontinui

In [3]:
def validate_dataset(recordings):
    # Number of subjects in each group
    for label in EXPECTED_SUBJECTS:
        n_subjects = sum(rec["label"] == label for rec in recordings)
        print(f"{label}: {n_subjects} subjects")

    # Check channel order
    reference_channels = recordings[0]["data"].ch_names

    if all(rec["data"].ch_names == reference_channels for rec in recordings):
        print(f"Channels consistent: {reference_channels}")
    else:
        print("Channel order differs between subjects.")



validate_dataset(recordings)

AD: 36 subjects
CN: 29 subjects
FTD: 23 subjects
Channels consistent: ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2']


## Connectivity extractor

In [6]:
from mne_connectivity import envelope_correlation

BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 45),
}

def crop_to_duration(raw, duration):
    
    raw = raw.copy()

    n_samples = int(duration * raw.info["sfreq"])
    actual_tmax = (n_samples - 1) / raw.info["sfreq"]

    return raw.crop(tmin=0, tmax=actual_tmax)

class ConnectivityExtractor:
    method = "aec"

    def __init__(
        self,
        epoch_length,
        target_duration,
        bands=BANDS,
        aec_orthogonalize="pairwise",
        aec_filter_kwargs=None,
        overlap=0.0,
    
    ):
        if overlap < 0 or overlap >= epoch_length:
            raise ValueError(
                f"overlap={overlap}s must be >= 0 and "
                f"< epoch_length={epoch_length}s"
            )

        self.epoch_length = epoch_length
        self.target_duration = target_duration
        self.overlap = overlap
        self.bands = bands
        self.aec_orthogonalize = aec_orthogonalize
        self.aec_filter_kwargs = aec_filter_kwargs or {}

    @property
    def config_name(self):
        return (
            f"aec_orth-{self.aec_orthogonalize}_"
            f"epoch{self.epoch_length}s_"
            f"ov{self.overlap}"
        )

    def metadata(self, ch_names, sfreq):
        """Metadata describing this configuration (saved with every file)."""
        return {
            "connectivity_method": self.method,
            "epoch_length_seconds": float(self.epoch_length),
            "overlap_seconds": float(self.overlap),
            "orthogonalization": self.aec_orthogonalize,
            "bands": {
                name: [float(low), float(high)]
                for name, (low, high) in self.bands.items()
            },
            "channel_names": list(ch_names),
            "sampling_frequency": float(sfreq),
        }
   
    def make_epochs(self, raw):

        raw = crop_to_duration(raw, self.target_duration)
        return mne.make_fixed_length_epochs(
            raw,
            duration=self.epoch_length,
            overlap=self.overlap,
            preload=True,
            verbose=False,
        )

    def extract_epochs(self, epochs):
        data = epochs.get_data(copy=True)
        sfreq = epochs.info["sfreq"]
        n_epochs, n_channels, _ = data.shape

        connectivity = np.zeros((n_epochs, len(self.bands), n_channels, n_channels))

        for band_idx, (l_freq, h_freq) in enumerate(self.bands.values()):
            filtered = mne.filter.filter_data(
                data, sfreq, l_freq, h_freq,
                verbose=False, **self.aec_filter_kwargs,
            )
            con = envelope_correlation(
                filtered,
                orthogonalize=self.aec_orthogonalize,
                absolute=True,
            )
            connectivity[:, band_idx] = con.get_data(output="dense").reshape(
                n_epochs, n_channels, n_channels
            )

        # Symmetrize (pairwise orthogonalization is asymmetric) and zero the diagonal
        connectivity = np.maximum(connectivity, np.swapaxes(connectivity, -1, -2))
        idx = np.arange(n_channels)
        connectivity[..., idx, idx] = 0

        return connectivity

## Subject selection, saving, and batch extraction

In [7]:
LABEL_TO_ID = {"AD": 1, "CN": 0, "FTD": 2}


def get_recordings_by_subject_range(recordings, start_subject, end_subject):
    if start_subject > end_subject:
        raise ValueError("start_subject must be less than or equal to end_subject.")

    selected = sorted(
        (rec for rec in recordings if start_subject <= rec["subject"] <= end_subject),
        key=lambda rec: rec["subject"],
    )
    if not selected:
        raise ValueError(
            f"No recordings found for subject range {start_subject} to {end_subject}."
        )
    return selected


def save_subject_connectivity(path, connectivity, subject, label, metadata):
    """Save one subject's connectivity array and metadata."""
    n_epochs = connectivity.shape[0]
    metadata = {
        **metadata,
        "subject": int(subject),
        "label": label,
        "label_id": int(LABEL_TO_ID[label]),
    }

    np.savez_compressed(
        path,
        X=connectivity.astype(np.float32),
        y=np.full(n_epochs, LABEL_TO_ID[label], dtype=np.int64),
        subject_ids=np.full(n_epochs, subject, dtype=np.int64),
        labels=np.full(n_epochs, label),
        epoch_numbers=np.arange(n_epochs, dtype=np.int64),
        metadata_json=np.array(json.dumps(metadata)),
    )


def process_recording(record, extractor, output_dir, overwrite=False):
    """Extract and save one subject. Returns ('saved' | 'skipped', path)."""
    subject, label = record["subject"], record["label"]
    path = output_dir / f"sub-{subject:03d}_label-{label}_{extractor.config_name}.npz"

    if path.exists() and not overwrite:
        return "skipped", path

    epochs = extractor.make_epochs(record["data"])
    connectivity = extractor.extract_epochs(epochs)
    metadata = extractor.metadata(epochs.ch_names, epochs.info["sfreq"])

    save_subject_connectivity(path, connectivity, subject, label, metadata)
    return "saved", path


def extract_and_save_subject_range(
    recordings, extractor, output_dir, start_subject, end_subject, overwrite=False
):
    """Extract and save connectivity for every subject in an inclusive ID range.

    Each subject goes to its own NPZ file immediately, so finished work
    survives if the run is interrupted.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    selected = get_recordings_by_subject_range(recordings, start_subject, end_subject)
    print(f"Processing {len(selected)} subject(s): {start_subject} to {end_subject}")

    results = {"saved": [], "skipped": [], "failed": []}

    for i, record in enumerate(selected, start=1):
        tag = f"[{i}/{len(selected)}] Subject {record['subject']:03d} ({record['label']})"
        try:
            status, path = process_recording(record, extractor, output_dir, overwrite)
            results[status].append(path)
            print(f"{tag}: {status}")
        except Exception as error:
            results["failed"].append({
                "subject": record["subject"],
                "label": record["label"],
                "error": str(error),
            })
            print(f"{tag}: FAILED - {error}")

    print(
        f"\nBatch complete. Saved: {len(results['saved'])} | "
        f"Skipped: {len(results['skipped'])} | Failed: {len(results['failed'])}"
    )
    return results

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "eeg_connectivity_dataset"

extractor = ConnectivityExtractor(epoch_length=4, overlap=0, target_duration=300)

In [9]:
results = extract_and_save_subject_range(
    recordings,
    extractor,
    output_dir=OUTPUT_DIR,
    start_subject=1,
    end_subject=65,
    overwrite=False,
)

Processing 65 subject(s): 1 to 65
[1/65] Subject 001 (AD): saved
[2/65] Subject 002 (AD): saved
[3/65] Subject 003 (AD): saved
[4/65] Subject 004 (AD): saved
[5/65] Subject 005 (AD): saved
[6/65] Subject 006 (AD): saved
[7/65] Subject 007 (AD): saved
[8/65] Subject 008 (AD): saved
[9/65] Subject 009 (AD): saved
[10/65] Subject 010 (AD): saved
[11/65] Subject 011 (AD): saved
[12/65] Subject 012 (AD): saved
[13/65] Subject 013 (AD): saved
[14/65] Subject 014 (AD): saved
[15/65] Subject 015 (AD): saved
[16/65] Subject 016 (AD): saved
[17/65] Subject 017 (AD): saved
[18/65] Subject 018 (AD): saved
[19/65] Subject 019 (AD): saved
[20/65] Subject 020 (AD): saved
[21/65] Subject 021 (AD): saved
[22/65] Subject 022 (AD): saved
[23/65] Subject 023 (AD): saved
[24/65] Subject 024 (AD): saved
[25/65] Subject 025 (AD): saved
[26/65] Subject 026 (AD): saved
[27/65] Subject 027 (AD): saved
[28/65] Subject 028 (AD): saved
[29/65] Subject 029 (AD): saved
[30/65] Subject 030 (AD): saved
[31/65] Subject

### Quality check

In [9]:
file_path = next(OUTPUT_DIR.glob(f"sub-030_*_{extractor.config_name}.npz"))

with np.load(file_path, allow_pickle=False) as data:
    X = data["X"]
    metadata = json.loads(data["metadata_json"].item())

print("X shape:", X.shape)
for key in ("subject", "label", "connectivity_method", "epoch_length_seconds",
            "bands", "channel_names"):
    print(f"{key}: {metadata[key]}")

X shape: (75, 5, 19, 19)
subject: 30
label: AD
connectivity_method: aec
epoch_length_seconds: 4.0
bands: {'delta': [1.0, 4.0], 'theta': [4.0, 8.0], 'alpha': [8.0, 13.0], 'beta': [13.0, 30.0], 'gamma': [30.0, 45.0]}
channel_names: ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2']


## Concatenate all files

In [10]:
ARRAY_KEYS = ("X", "y", "subject_ids", "labels", "epoch_numbers")
CONSISTENT_KEYS = (
    "connectivity_method", "epoch_length_seconds", "overlap_seconds",
    "orthogonalization", "bands", "channel_names", "sampling_frequency",
)


def concatenate_connectivity_files(input_dir, output_file, extractor):
    """Combine subject-level NPZ files (matching the extractor config) into one dataset."""
    input_dir, output_file = Path(input_dir), Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    pattern = f"sub-*_label-*_{extractor.config_name}.npz"
    file_paths = sorted(input_dir.glob(pattern))
    if not file_paths:
        raise FileNotFoundError(f"No files matched pattern: {pattern}")

    arrays = {key: [] for key in ARRAY_KEYS}
    reference = None

    for file_path in file_paths:
        with np.load(file_path, allow_pickle=False) as data:
            metadata = json.loads(data["metadata_json"].item())

            if reference is None:
                reference = metadata
            else:
                mismatched = [k for k in CONSISTENT_KEYS if metadata[k] != reference[k]]
                if mismatched:
                    raise ValueError(
                        f"Incompatible metadata in {file_path.name}: {mismatched}"
                    )

            for key in ARRAY_KEYS:
                arrays[key].append(data[key])

    arrays = {key: np.concatenate(parts) for key, parts in arrays.items()}

    combined_metadata = {key: reference[key] for key in CONSISTENT_KEYS}
    combined_metadata.update({
        "label_mapping": LABEL_TO_ID,
        "n_subjects": int(len(np.unique(arrays["subject_ids"]))),
        "n_epochs": int(arrays["X"].shape[0]),
        "matrix_shape": list(arrays["X"].shape[1:]),
    })

    np.savez_compressed(
        output_file,
        X=arrays["X"].astype(np.float32),
        y=arrays["y"].astype(np.int64),
        subject_ids=arrays["subject_ids"].astype(np.int64),
        labels=arrays["labels"],
        epoch_numbers=arrays["epoch_numbers"].astype(np.int64),
        metadata_json=np.array(json.dumps(combined_metadata)),
    )

    print(f"Saved combined dataset: {output_file.name}")
    print("X shape:", arrays["X"].shape)
    print("y shape:", arrays["y"].shape)
    print("Unique subjects:", combined_metadata["n_subjects"])
    return output_file

In [11]:
combined_file = concatenate_connectivity_files(
    input_dir=OUTPUT_DIR,
    output_file=OUTPUT_DIR / "concatenated" / f"all_subjects_connectivity.npz",
    extractor=extractor,
)

Saved combined dataset: all_subjects_connectivity.npz
X shape: (4875, 5, 19, 19)
y shape: (4875,)
Unique subjects: 65
